# 06D — Local Pixel-Hole Repair for Station Extraction

This notebook follows the 06C diagnostic result.

It repairs only **isolated station-level NoData pixels** when nearby native pixels are valid.

### Safeguards
- No raster-wide filling.
- No edge clamping.
- Outside-coverage stations remain `NaN`.
- Exact valid source pixel is always preferred.
- For an isolated NoData source pixel, the notebook searches 3×3 first, then 5×5.
- A repair is used only when at least 3 valid neighboring native pixels exist.
- Every repaired value is written to a separate audit log.
- The final output is `station_samples_native_tidy_REPAIRED.csv`.

Run this after **06C** and before the final **07** modelling notebook.


In [1]:

# ============================================================
# 06D — LOCAL PIXEL-HOLE REPAIR FOR STATION EXTRACTION
# Khulna Precipitation Downscaling
#
# Purpose:
# - Repair ONLY isolated station-level NoData pixels when nearby
#   native source pixels are valid.
# - Never fill entire rasters.
# - Never edge-clamp stations.
# - Preserve true outside-coverage cases as NaN.
# ============================================================

from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import rasterio
from pyproj import Transformer


# ============================================================
# 1. FIND PROJECT ROOT
# ============================================================

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "data").exists():
            return candidate

    raise FileNotFoundError(
        "Project root not found. "
        "Run this notebook from inside the repository."
    )


PROJECT_ROOT = find_project_root()

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("PROJECT_ROOT =", PROJECT_ROOT)


# ============================================================
# 2. INPUT / OUTPUT PATHS
# ============================================================

GAUGE_PATH = PROCESSED_DIR / "gauge_monthly_clean.csv"

if not GAUGE_PATH.exists():
    raise FileNotFoundError(
        "gauge_monthly_clean.csv not found. "
        "Run Notebook 02 first."
    )

OUTPUT_SAMPLES = (
    PROCESSED_DIR
    / "station_samples_native_tidy_REPAIRED.csv"
)

OUTPUT_QC = (
    PROCESSED_DIR
    / "station_extraction_qc_REPAIRED.csv"
)

OUTPUT_REPAIR_LOG = (
    PROCESSED_DIR
    / "station_local_pixel_repair_log.csv"
)

OUTPUT_MISSING_QC = (
    PROCESSED_DIR
    / "station_extraction_missing_qc_REPAIRED.csv"
)

OUTPUT_STATIC_QC = (
    PROCESSED_DIR
    / "station_static_extraction_qc_REPAIRED.csv"
)


# ============================================================
# 3. SETTINGS
# ============================================================

WGS84_PROJ = "+proj=longlat +datum=WGS84 +no_defs"

# Local neighborhood search radii.
# Radius=1 means 3x3, radius=2 means 5x5.
# We allow 5x5 maximum because 06C showed valid nearby cells.
MAX_RADIUS = 2

# Minimum number of valid neighboring cells required before repair.
MIN_VALID_NEIGHBORS = 3

# Repair method.
# "mean" is conservative and simple for isolated pixel holes.
REPAIR_METHOD = "mean"


# ============================================================
# 4. ACTUAL SOURCE FOLDER MAPPING
# ============================================================

PRECIP_FOLDERS = {
    "CCS": "CCS",
    "PDIR": "PDIR",
    "GSMaP_MVK": "GSMaP_MVK",
    "CDR": "CDR",
    "CHIRPS": "CHIRPS_TIFF_2017_2022",
    "IMERG": "IMERG_Monthly",
    "GSMaP_Gauge": "GSMaP_Gauge_v7",
    "ERA5": "ERA5_TIFF",
}

DYNAMIC_LAND = [
    "NDVI",
    "LST_Day",
]


# ============================================================
# 5. LOAD GAUGE DATA
# ============================================================

gauge = pd.read_csv(
    GAUGE_PATH
)

required_cols = [
    "station_id",
    "year",
    "month",
    "rainfall_mm",
    "latitude",
    "longitude",
]

missing = [
    c for c in required_cols
    if c not in gauge.columns
]

if missing:
    raise KeyError(
        f"Gauge file missing required columns: {missing}"
    )

print("Gauge rows:", len(gauge))
print("Stations:", gauge["station_id"].nunique())


# ============================================================
# 6. HELPERS
# ============================================================

def parse_ym(name):
    stem = Path(name).stem

    patterns = [
        r"(?<!\d)(20\d{2})[_-](0?[1-9]|1[0-2])(?!\d)",
        r"(?<!\d)(20\d{2})(0[1-9]|1[0-2])(?!\d)",
    ]

    for pat in patterns:
        m = re.search(
            pat,
            stem
        )

        if m:
            return (
                int(m.group(1)),
                int(m.group(2))
            )

    return None


def list_rasters(folder):
    if not folder.exists():
        return []

    return sorted([
        *folder.rglob("*.tif"),
        *folder.rglob("*.tiff"),
    ])


def monthly_map(folder):
    out = {}

    for p in list_rasters(folder):
        ym = parse_ym(
            p.name
        )

        if ym is None:
            continue

        if ym in out:
            raise ValueError(
                f"Duplicate raster for {folder.name} {ym}:\n"
                f"{out[ym]}\n{p}"
            )

        out[ym] = p

    return out


def choose_static(
    folder_name,
    preferred_names
):
    folder = (
        RAW_DIR
        / "predictors"
        / folder_name
    )

    files = list_rasters(
        folder
    )

    if not files:
        raise FileNotFoundError(
            f"No raster found for {folder_name}"
        )

    lookup = {
        p.name.lower(): p
        for p in files
    }

    for name in preferred_names:
        if name.lower() in lookup:
            return lookup[
                name.lower()
            ]

    clean = [
        p for p in files
        if not any(
            x in p.stem.lower()
            for x in [
                "clip",
                "tmp",
                "temp",
                "aligned",
                "resampl",
            ]
        )
    ]

    if len(clean) == 1:
        return clean[0]

    if len(files) == 1:
        return files[0]

    raise ValueError(
        f"Ambiguous static predictor {folder_name}:\n"
        + "\n".join(
            str(p)
            for p in files
        )
    )


def bounds_look_geographic(bounds):
    return (
        -180 <= bounds.left <= 180
        and -180 <= bounds.right <= 180
        and -90 <= bounds.bottom <= 90
        and -90 <= bounds.top <= 90
    )


def station_xy_for_raster(
    src,
    lon,
    lat
):
    """
    Return station x/y in raster CRS.

    If raster bounds are clearly geographic, use lon/lat directly.
    This safely handles the CDR EngineeringCRS metadata issue.
    """

    b = src.bounds

    if bounds_look_geographic(
        b
    ):
        return (
            float(lon),
            float(lat),
            "direct_geographic"
        )

    if src.crs is None:
        return (
            None,
            None,
            "crs_missing_non_geographic"
        )

    try:
        transformer = Transformer.from_crs(
            WGS84_PROJ,
            src.crs,
            always_xy=True
        )

        x, y = transformer.transform(
            float(lon),
            float(lat)
        )

        return (
            x,
            y,
            "transformed"
        )

    except Exception as e:
        warnings.warn(
            f"Coordinate transformation failed: {e}"
        )

        return (
            None,
            None,
            "transform_failed"
        )


def value_from_masked_pixel(
    pixel,
    nodata
):
    """
    Convert one masked-array pixel to float/NaN safely.
    """

    if np.ma.is_masked(
        pixel
    ):
        return np.nan

    try:
        value = float(
            pixel
        )
    except Exception:
        return np.nan

    if not np.isfinite(
        value
    ):
        return np.nan

    if (
        nodata is not None
        and np.isclose(
            value,
            nodata
        )
    ):
        return np.nan

    return value


def local_valid_neighbors(
    src,
    row,
    col,
    radius
):
    """
    Return valid neighboring native pixels around a station pixel.
    The center pixel itself is excluded.
    """

    r0 = max(
        0,
        row - radius
    )

    r1 = min(
        src.height,
        row + radius + 1
    )

    c0 = max(
        0,
        col - radius
    )

    c1 = min(
        src.width,
        col + radius + 1
    )

    arr = src.read(
        1,
        window=(
            (r0, r1),
            (c0, c1)
        ),
        masked=True
    )

    # Important for integer rasters:
    # convert to float before filling with NaN.
    arr_float = arr.astype(
        "float64"
    )

    filled = np.asarray(
        arr_float.filled(
            np.nan
        ),
        dtype=float
    )

    # Exclude center cell from neighbor calculation.
    center_r = row - r0
    center_c = col - c0

    if (
        0 <= center_r < filled.shape[0]
        and 0 <= center_c < filled.shape[1]
    ):
        filled[
            center_r,
            center_c
        ] = np.nan

    valid_values = filled[
        np.isfinite(
            filled
        )
    ]

    # Explicitly remove numeric nodata values if necessary.
    if src.nodata is not None:
        valid_values = valid_values[
            ~np.isclose(
                valid_values,
                src.nodata
            )
        ]

    return valid_values


def repair_from_neighbors(
    values
):
    if len(values) == 0:
        return np.nan

    if REPAIR_METHOD == "mean":
        return float(
            np.nanmean(
                values
            )
        )

    elif REPAIR_METHOD == "median":
        return float(
            np.nanmedian(
                values
            )
        )

    raise ValueError(
        f"Unknown REPAIR_METHOD: {REPAIR_METHOD}"
    )


def sample_native_with_local_repair(
    path,
    lon,
    lat,
    allow_repair=True
):
    """
    Native source sampling with local pixel-hole repair.

    Rules:
    1. Exact valid pixel -> use exact value.
    2. Outside source bounds -> NaN, no repair.
    3. Exact pixel NoData -> search 3x3, then 5x5.
    4. Repair only if enough valid neighboring pixels exist.
    5. Never modify the raster itself.
    """

    with rasterio.open(
        path
    ) as src:

        x, y, coord_status = (
            station_xy_for_raster(
                src,
                lon,
                lat
            )
        )

        if x is None or y is None:
            return (
                np.nan,
                coord_status,
                {
                    "repair_used": False,
                    "radius": np.nan,
                    "neighbor_count": 0,
                }
            )

        b = src.bounds

        if not (
            b.left <= x <= b.right
            and b.bottom <= y <= b.top
        ):
            return (
                np.nan,
                "outside",
                {
                    "repair_used": False,
                    "radius": np.nan,
                    "neighbor_count": 0,
                }
            )

        try:
            row, col = src.index(
                x,
                y
            )
        except Exception:
            return (
                np.nan,
                "index_failed",
                {
                    "repair_used": False,
                    "radius": np.nan,
                    "neighbor_count": 0,
                }
            )

        if (
            row < 0
            or row >= src.height
            or col < 0
            or col >= src.width
        ):
            return (
                np.nan,
                "outside",
                {
                    "repair_used": False,
                    "radius": np.nan,
                    "neighbor_count": 0,
                }
            )

        exact = src.read(
            1,
            window=(
                (row, row + 1),
                (col, col + 1)
            ),
            masked=True
        )

        if exact.size == 0:
            exact_value = np.nan
        else:
            exact_value = (
                value_from_masked_pixel(
                    exact[0, 0],
                    src.nodata
                )
            )

        # ----------------------------------------------------
        # Exact pixel valid
        # ----------------------------------------------------
        if np.isfinite(
            exact_value
        ):
            return (
                exact_value,
                "ok_exact",
                {
                    "repair_used": False,
                    "radius": 0,
                    "neighbor_count": 0,
                }
            )

        # ----------------------------------------------------
        # Repair not allowed
        # ----------------------------------------------------
        if not allow_repair:
            return (
                np.nan,
                "nodata",
                {
                    "repair_used": False,
                    "radius": np.nan,
                    "neighbor_count": 0,
                }
            )

        # ----------------------------------------------------
        # Search locally: 3x3 then 5x5
        # ----------------------------------------------------
        for radius in range(
            1,
            MAX_RADIUS + 1
        ):

            neighbors = (
                local_valid_neighbors(
                    src,
                    row,
                    col,
                    radius
                )
            )

            if len(
                neighbors
            ) >= MIN_VALID_NEIGHBORS:

                repaired = (
                    repair_from_neighbors(
                        neighbors
                    )
                )

                if np.isfinite(
                    repaired
                ):
                    return (
                        repaired,
                        "repaired_local_neighbors",
                        {
                            "repair_used": True,
                            "radius": radius,
                            "neighbor_count": int(
                                len(
                                    neighbors
                                )
                            ),
                            "neighbor_min": float(
                                np.nanmin(
                                    neighbors
                                )
                            ),
                            "neighbor_max": float(
                                np.nanmax(
                                    neighbors
                                )
                            ),
                            "neighbor_mean": float(
                                np.nanmean(
                                    neighbors
                                )
                            ),
                        }
                    )

        # ----------------------------------------------------
        # No defensible local repair
        # ----------------------------------------------------
        return (
            np.nan,
            "nodata_no_local_repair",
            {
                "repair_used": False,
                "radius": np.nan,
                "neighbor_count": 0,
            }
        )


# ============================================================
# 7. PARSER SELF-TEST
# ============================================================

parser_tests = {
    "CCS_2022_01.tif": (2022, 1),
    "abc-2021-12.tif": (2021, 12),
    "IMERG_202203.tif": (2022, 3),
}

for test_name, expected in parser_tests.items():
    got = parse_ym(
        test_name
    )

    if got != expected:
        raise RuntimeError(
            f"Parser failed for {test_name}: "
            f"got {got}, expected {expected}"
        )

print(
    "Filename year-month parser: OK"
)


# ============================================================
# 8. CANONICAL STATIC SOURCES
# ============================================================

DEM_PATH = choose_static(
    "DEM",
    [
        "Khulna_SRTM_DEM.tif",
        "DEM.tif",
    ]
)

DISTANCE_PATH = choose_static(
    "Distance_Sea",
    [
        "Distance_Sea.tif",
        "distance_to_sea.tif",
    ]
)

print("\nCanonical DEM:")
print(DEM_PATH)

print("\nCanonical Distance_Sea:")
print(DISTANCE_PATH)


# ============================================================
# 9. BUILD SOURCE MAPS
# ============================================================

precip_maps = {}

print(
    "\n=========================================="
)

print(
    "PRECIPITATION SOURCE CHECK"
)

print(
    "=========================================="
)

for feature_name, folder_name in (
    PRECIP_FOLDERS.items()
):

    folder = (
        RAW_DIR
        / "precipitation"
        / folder_name
    )

    if not folder.exists():
        raise FileNotFoundError(
            f"Missing precipitation folder:\n{folder}"
        )

    mm = monthly_map(
        folder
    )

    precip_maps[
        feature_name
    ] = mm

    print(
        f"{feature_name:14s} | "
        f"folder={folder_name:24s} | "
        f"parsed={len(mm)}"
    )

    missing_months = [
        (y, m)
        for y in range(
            2017,
            2023
        )
        for m in range(
            1,
            13
        )
        if (
            y,
            m
        ) not in mm
    ]

    if missing_months:
        raise FileNotFoundError(
            f"{feature_name} missing months:\n"
            f"{missing_months}"
        )


land_maps = {}

print(
    "\n=========================================="
)

print(
    "DYNAMIC LAND SOURCE CHECK"
)

print(
    "=========================================="
)

for feature_name in DYNAMIC_LAND:

    folder = (
        RAW_DIR
        / "predictors"
        / feature_name
    )

    if not folder.exists():
        raise FileNotFoundError(
            f"Missing predictor folder:\n{folder}"
        )

    mm = monthly_map(
        folder
    )

    land_maps[
        feature_name
    ] = mm

    print(
        f"{feature_name:14s} | "
        f"parsed={len(mm)}"
    )

    missing_months = [
        (y, m)
        for y in range(
            2017,
            2023
        )
        for m in range(
            1,
            13
        )
        if (
            y,
            m
        ) not in mm
    ]

    if missing_months:
        raise FileNotFoundError(
            f"{feature_name} missing months:\n"
            f"{missing_months}"
        )


# ============================================================
# 10. STATIC STATION EXTRACTION WITH LOCAL REPAIR
# ============================================================

station_static = {}
static_qc_rows = []
repair_log_rows = []


for station, s in gauge.groupby(
    "station_id",
    sort=False
):

    lon = float(
        s["longitude"].median()
    )

    lat = float(
        s["latitude"].median()
    )

    dem, dem_status, dem_meta = (
        sample_native_with_local_repair(
            DEM_PATH,
            lon,
            lat,
            allow_repair=True
        )
    )

    dfs, dfs_status, dfs_meta = (
        sample_native_with_local_repair(
            DISTANCE_PATH,
            lon,
            lat,
            allow_repair=True
        )
    )

    station_static[
        station
    ] = {
        "DEM": dem,
        "Distance_Sea": dfs,
    }

    static_qc_rows.append(
        {
            "station_id": station,
            "DEM_status": dem_status,
            "Distance_Sea_status": dfs_status,
        }
    )

    for feature, value, status, meta, path in [
        (
            "DEM",
            dem,
            dem_status,
            dem_meta,
            DEM_PATH
        ),
        (
            "Distance_Sea",
            dfs,
            dfs_status,
            dfs_meta,
            DISTANCE_PATH
        ),
    ]:

        if meta.get(
            "repair_used",
            False
        ):
            repair_log_rows.append(
                {
                    "station_id": station,
                    "year": np.nan,
                    "month": np.nan,
                    "feature": feature,
                    "source": str(path),
                    "repaired_value": value,
                    "status": status,
                    **meta,
                }
            )


static_qc_df = pd.DataFrame(
    static_qc_rows
)

print(
    "\nStatic predictor extraction status:"
)

display(
    static_qc_df
)


# ============================================================
# 11. BUILD REPAIRED TIDY TABLE
# ============================================================

records = []
qc_rows = []


for _, r in gauge.iterrows():

    station = r[
        "station_id"
    ]

    y = int(
        r[
            "year"
        ]
    )

    m = int(
        r[
            "month"
        ]
    )

    lon = float(
        r[
            "longitude"
        ]
    )

    lat = float(
        r[
            "latitude"
        ]
    )

    rec = {
        "station_id": station,
        "year": y,
        "month": m,
        "date": f"{y}-{m:02d}-01",
        "latitude": lat,
        "longitude": lon,
        "rainfall_mm": float(
            r[
                "rainfall_mm"
            ]
        ),
    }

    rec.update(
        station_static[
            station
        ]
    )

    # --------------------------------------------------------
    # Precipitation features
    # --------------------------------------------------------
    for feature_name in (
        PRECIP_FOLDERS.keys()
    ):

        path = precip_maps[
            feature_name
        ].get(
            (
                y,
                m
            )
        )

        if path is None:
            value = np.nan
            status = (
                "missing_file"
            )
            meta = {
                "repair_used": False,
                "radius": np.nan,
                "neighbor_count": 0,
            }

        else:
            value, status, meta = (
                sample_native_with_local_repair(
                    path,
                    lon,
                    lat,
                    allow_repair=True
                )
            )

        # Precipitation cannot be negative.
        if (
            np.isfinite(
                value
            )
            and value < 0
        ):
            value = np.nan
            status = (
                "negative_invalid"
            )

        rec[
            feature_name
        ] = value

        qc_rows.append(
            {
                "station_id": station,
                "year": y,
                "month": m,
                "feature": feature_name,
                "status": status,
                "source": str(path)
                if path is not None
                else None,
                "repair_used": meta.get(
                    "repair_used",
                    False
                ),
                "radius": meta.get(
                    "radius",
                    np.nan
                ),
                "neighbor_count": meta.get(
                    "neighbor_count",
                    0
                ),
            }
        )

        if meta.get(
            "repair_used",
            False
        ):
            repair_log_rows.append(
                {
                    "station_id": station,
                    "year": y,
                    "month": m,
                    "feature": feature_name,
                    "source": str(path),
                    "repaired_value": value,
                    "status": status,
                    **meta,
                }
            )

    # --------------------------------------------------------
    # Dynamic land
    # --------------------------------------------------------
    for feature_name in (
        DYNAMIC_LAND
    ):

        path = land_maps[
            feature_name
        ].get(
            (
                y,
                m
            )
        )

        if path is None:
            value = np.nan
            status = (
                "missing_file"
            )
            meta = {
                "repair_used": False,
                "radius": np.nan,
                "neighbor_count": 0,
            }

        else:
            value, status, meta = (
                sample_native_with_local_repair(
                    path,
                    lon,
                    lat,
                    allow_repair=True
                )
            )

        rec[
            feature_name
        ] = value

        qc_rows.append(
            {
                "station_id": station,
                "year": y,
                "month": m,
                "feature": feature_name,
                "status": status,
                "source": str(path)
                if path is not None
                else None,
                "repair_used": meta.get(
                    "repair_used",
                    False
                ),
                "radius": meta.get(
                    "radius",
                    np.nan
                ),
                "neighbor_count": meta.get(
                    "neighbor_count",
                    0
                ),
            }
        )

        if meta.get(
            "repair_used",
            False
        ):
            repair_log_rows.append(
                {
                    "station_id": station,
                    "year": y,
                    "month": m,
                    "feature": feature_name,
                    "source": str(path),
                    "repaired_value": value,
                    "status": status,
                    **meta,
                }
            )

    records.append(
        rec
    )


# ============================================================
# 12. OUTPUT TABLES
# ============================================================

samples = pd.DataFrame(
    records
).sort_values(
    [
        "station_id",
        "year",
        "month",
    ]
).reset_index(
    drop=True
)

qc = pd.DataFrame(
    qc_rows
)

repair_log = pd.DataFrame(
    repair_log_rows
)


print(
    "\n=========================================="
)

print(
    "LOCAL PIXEL-HOLE REPAIR COMPLETE"
)

print(
    "=========================================="
)

print(
    "Repaired modelling table:",
    samples.shape
)

display(
    samples.head()
)


# ============================================================
# 13. MISSING-VALUE QC
# ============================================================

FEATURES = (
    list(
        PRECIP_FOLDERS.keys()
    )
    + [
        "DEM",
        "NDVI",
        "LST_Day",
        "Distance_Sea",
    ]
)

missing_qc = pd.DataFrame(
    {
        "missing_count":
            samples[
                FEATURES
            ].isna().sum(),

        "missing_pct":
            samples[
                FEATURES
            ].isna().mean()
            * 100.0,
    }
).sort_values(
    "missing_pct",
    ascending=False
)


print(
    "\n=========================================="
)

print(
    "MISSING VALUES AFTER LOCAL REPAIR"
)

print(
    "=========================================="
)

display(
    missing_qc
)


# ============================================================
# 14. REPAIR SUMMARY
# ============================================================

print(
    "\n=========================================="
)

print(
    "LOCAL REPAIR SUMMARY"
)

print(
    "=========================================="
)

if len(
    repair_log
) > 0:

    repair_summary = (
        repair_log
        .groupby(
            [
                "feature",
                "status",
                "radius",
            ],
            dropna=False
        )
        .size()
        .rename(
            "n"
        )
        .reset_index()
    )

    display(
        repair_summary
    )

    print(
        "\nTotal locally repaired values:",
        len(
            repair_log
        )
    )

else:

    print(
        "No local repairs were used."
    )


# ============================================================
# 15. REMAINING PROBLEM STATUS
# ============================================================

status_summary = (
    qc
    .groupby(
        [
            "feature",
            "status"
        ]
    )
    .size()
    .rename(
        "n"
    )
    .reset_index()
)


print(
    "\n=========================================="
)

print(
    "EXTRACTION STATUS AFTER REPAIR"
)

print(
    "=========================================="
)

display(
    status_summary
)


remaining_problem = qc[
    ~qc[
        "status"
    ].isin(
        [
            "ok_exact",
            "repaired_local_neighbors",
        ]
    )
].copy()


print(
    "\n=========================================="
)

print(
    "REMAINING UNRESOLVED EXTRACTIONS"
)

print(
    "=========================================="
)


if len(
    remaining_problem
) == 0:

    print(
        "None. All station-feature records are now usable."
    )

else:

    display(
        remaining_problem
    )


# ============================================================
# 16. COMPLETE-CASE MODEL READINESS
# ============================================================

PRECIP8 = [
    "CCS",
    "PDIR",
    "GSMaP_MVK",
    "CDR",
    "CHIRPS",
    "IMERG",
    "GSMaP_Gauge",
    "ERA5",
]

LAND = [
    "DEM",
    "NDVI",
    "LST_Day",
    "Distance_Sea",
]

COMBINATIONS = {
    "Comb1": PRECIP8,
    "Comb1_land": PRECIP8 + LAND,
    "Comb2": [
        "CHIRPS",
        "CDR",
        "ERA5",
    ],
    "Comb2_land": [
        "CHIRPS",
        "CDR",
        "ERA5",
    ] + LAND,
}

SPLITS = {
    "train": [
        2017,
        2018,
        2019,
        2020,
    ],
    "validation": [
        2021,
    ],
    "test": [
        2022,
    ],
}

readiness_rows = []

for combo, features in (
    COMBINATIONS.items()
):

    for split, years in (
        SPLITS.items()
    ):

        d = samples[
            samples[
                "year"
            ].isin(
                years
            )
        ].copy()

        total = len(
            d
        )

        complete = d.dropna(
            subset=[
                "rainfall_mm"
            ]
            + features
        )

        readiness_rows.append(
            {
                "combination": combo,
                "split": split,
                "total_rows": total,
                "complete_rows": len(
                    complete
                ),
                "dropped_rows":
                    total
                    - len(
                        complete
                    ),
                "complete_pct":
                    (
                        100.0
                        * len(
                            complete
                        )
                        / total
                    )
                    if total
                    else np.nan,
                "stations_remaining":
                    complete[
                        "station_id"
                    ].nunique(),
            }
        )


readiness = pd.DataFrame(
    readiness_rows
)


print(
    "\n=========================================="
)

print(
    "MODEL READINESS AFTER LOCAL REPAIR"
)

print(
    "=========================================="
)

display(
    readiness
)


# ============================================================
# 17. SAVE OUTPUTS
# ============================================================

samples.to_csv(
    OUTPUT_SAMPLES,
    index=False
)

qc.to_csv(
    OUTPUT_QC,
    index=False
)

repair_log.to_csv(
    OUTPUT_REPAIR_LOG,
    index=False
)

missing_qc.to_csv(
    OUTPUT_MISSING_QC
)

static_qc_df.to_csv(
    OUTPUT_STATIC_QC,
    index=False
)

readiness.to_csv(
    PROCESSED_DIR
    / "model_readiness_after_local_repair.csv",
    index=False
)


# ============================================================
# 18. FINAL SUMMARY
# ============================================================

print(
    "\n=========================================="
)

print(
    "06D LOCAL PIXEL-HOLE REPAIR DONE"
)

print(
    "=========================================="
)

print(
    "Saved repaired modelling table:",
    OUTPUT_SAMPLES
)

print(
    "Saved extraction QC:",
    OUTPUT_QC
)

print(
    "Saved local repair log:",
    OUTPUT_REPAIR_LOG
)

print(
    "Saved missing-value QC:",
    OUTPUT_MISSING_QC
)

print(
    "Saved model readiness:",
    PROCESSED_DIR
    / "model_readiness_after_local_repair.csv"
)

print(
    "\nIMPORTANT:"
)

print(
    "Only station-level isolated NoData pixels were repaired "
    "using nearby native source pixels."
)

print(
    "No raster-wide filling was performed."
)

print(
    "Stations outside source coverage remain NaN."
)

print(
    "Review MODEL READINESS AFTER LOCAL REPAIR before Notebook 07."
)


PROJECT_ROOT = E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh
Gauge rows: 432
Stations: 6
Filename year-month parser: OK

Canonical DEM:
E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\raw\predictors\DEM\Khulna_SRTM_DEM.tif

Canonical Distance_Sea:
E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\raw\predictors\Distance_Sea\Distance_Sea.tif

PRECIPITATION SOURCE CHECK
CCS            | folder=CCS                      | parsed=72
PDIR           | folder=PDIR                     | parsed=72
GSMaP_MVK      | folder=GSMaP_MVK                | parsed=72
CDR            | folder=CDR                      | parsed=72
CHIRPS         | folder=CHIRPS_TIFF_2017_2022    | parsed=72
IMERG          | folder=IMERG_Monthly            | parsed=72
GSMaP_Gauge  

,station_id,DEM_status,Distance_Sea_status
0,CL503,ok_exact,ok_exact
1,CL504,ok_exact,ok_exact
2,CL509,ok_exact,repaired_local_neighbors
3,CL510,ok_exact,ok_exact
4,CL515,ok_exact,ok_exact
5,CL517,ok_exact,ok_exact



LOCAL PIXEL-HOLE REPAIR COMPLETE
Repaired modelling table: (432, 19)


,station_id,year,month,date,latitude,longitude,rainfall_mm,DEM,Distance_Sea,CCS,PDIR,GSMaP_MVK,CDR,CHIRPS,IMERG,GSMaP_Gauge,ERA5,NDVI,LST_Day
0,CL503,2017,1,2017-01-01,22.6012,89.5195,0.0,5.0,32.322636,0.0,1.0,1.966016,5.468657,4.258131,0.000,0.000000,0.208773,0.3041,23.28
1,CL503,2017,2,2017-02-01,22.6012,89.5195,0.0,5.0,32.322636,0.0,0.0,0.000000,0.000000,5.859640,0.003,0.193933,3.796070,0.3495,27.74
2,CL503,2017,3,2017-03-01,22.6012,89.5195,345.0,5.0,32.322636,2.0,17.0,49.963665,39.437916,47.325570,0.097,117.178435,101.051186,0.3115,29.21
3,CL503,2017,4,2017-04-01,22.6012,89.5195,270.0,5.0,32.322636,31.0,56.0,334.475193,98.197905,93.311977,0.155,132.086874,85.262344,0.3005,31.95
4,CL503,2017,5,2017-05-01,22.6012,89.5195,783.0,5.0,32.322636,60.0,122.0,134.737306,119.039114,149.536093,0.166,152.076353,107.824173,0.4503,32.39



MISSING VALUES AFTER LOCAL REPAIR


...


REMAINING UNRESOLVED EXTRACTIONS


,station_id,year,month,feature,status,source,repair_used,radius,neighbor_count
69,CL503,2017,7,LST_Day,nodata_no_local_repair,E:\Geospatial\Precipitation-Downscaling-Khulna...,False,NaN,0
72,CL503,2017,8,GSMaP_MVK,outside,E:\Geospatial\Precipitation-Downscaling-Khulna...,False,NaN,0
192,CL503,2018,8,GSMaP_MVK,outside,E:\Geospatial\Precipitation-Downscaling-Khulna...,False,NaN,0
199,CL503,2018,8,LST_Day,nodata_no_local_repair,E:\Geospatial\Precipitation-Downscaling-Khulna...,False,NaN,0
312,CL503,2019,8,GSMaP_MVK,outside,E:\Geospatial\Precipitation-Downscaling-Khulna...,False,NaN,0
432,CL503,2020,8,GSMaP_MVK,outside,E:\Geospatial\Precipitation-Downscaling-Khulna...,False,NaN,0
549,CL503,2021,7,LST_Day,nodata_no_local_repair,E:\Geospatial\Precipitation-Downscaling-Khulna...,False,NaN,0
552,CL503,2021,8,GSMaP_MVK,outside,E:\Geospatial\Precipitation-Downscaling-Khulna...,False,NaN,0
559,CL503,2021,8,LST_Day,nodata_no_local_repair,E:\Geospatial\Precipitation-Downscaling-Khulna...,False,NaN,0
659,CL503,2022,6,LST_Day,nodata_no_local_repair,E:\Geospatial\Precipitation-Downscaling-Khulna...,False,NaN,0



MODEL READINESS AFTER LOCAL REPAIR


,combination,split,total_rows,complete_rows,dropped_rows,complete_pct,stations_remaining
0,Comb1,train,288,276,12,95.833333,6
1,Comb1,validation,72,69,3,95.833333,6
2,Comb1,test,72,69,3,95.833333,6
3,Comb1_land,train,288,262,26,90.972222,6
4,Comb1_land,validation,72,63,9,87.500000,6
5,Comb1_land,test,72,64,8,88.888889,6
6,Comb2,train,288,288,0,100.000000,6
7,Comb2,validation,72,72,0,100.000000,6
8,Comb2,test,72,72,0,100.000000,6
9,Comb2_land,train,288,273,15,94.791667,6



06D LOCAL PIXEL-HOLE REPAIR DONE
Saved repaired modelling table: E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\processed\station_samples_native_tidy_REPAIRED.csv
Saved extraction QC: E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\processed\station_extraction_qc_REPAIRED.csv
Saved local repair log: E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\processed\station_local_pixel_repair_log.csv
Saved missing-value QC: E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\processed\station_extraction_missing_qc_REPAIRED.csv
Saved model readiness: E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District